# Gold: lab_test_trends

Builds `<catalog>.gold.lab_test_trends` (one row per test_name per calendar month) from `silver.lab_results`, with optional calendar enrichment and LAG-based month-over-month abnormal-rate trend. Full rebuild per STTM `sttm_silver_to_gold.xlsx`.

In [0]:
import re

dbutils.widgets.text("catalog", "adb_carsync", "Catalog")
catalog = dbutils.widgets.get("catalog").strip()

if not re.fullmatch(r"[A-Za-z0-9_]+", catalog):
    raise ValueError(f"Invalid catalog name: {catalog}")

target_table = f"{catalog}.gold.lab_test_trends"
print(f"Target: {target_table}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {target_table} AS
WITH monthly AS (
  SELECT
    test_name,
    YEAR(test_date)                       AS year,
    MONTH(test_date)                      AS month,
    -- first day of the month as yyyyMMdd key for the calendar join
    CAST(DATE_FORMAT(DATE_TRUNC('MONTH', MIN(test_date)), 'yyyyMMdd') AS INT) AS month_date_key,
    COUNT(*)                              AS total_tests,
    COUNT_IF(result_status = 'Abnormal')  AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')  AS critical_test_count
  FROM {catalog}.silver.lab_results
  GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
rates AS (
  SELECT *,
         CAST(abnormal_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS abnormal_rate_pct,
         CAST(critical_test_count * 100.0 / total_tests AS DECIMAL(5,2)) AS critical_rate_pct
  FROM monthly
),
with_lag AS (
  SELECT *,
         LAG(abnormal_rate_pct) OVER (PARTITION BY test_name ORDER BY year, month) AS prev_month_abnormal_rate_pct
  FROM rates
)
SELECT
  r.test_name,
  CAST(r.year AS INT)                 AS year,
  CAST(r.month AS INT)                AS month,
  c.month_name,
  c.quarter_name,
  CAST(r.total_tests AS INT)          AS total_tests,
  CAST(r.abnormal_test_count AS INT)  AS abnormal_test_count,
  CAST(r.critical_test_count AS INT)  AS critical_test_count,
  r.abnormal_rate_pct,
  r.critical_rate_pct,
  CAST(r.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                       AS prev_month_abnormal_rate_pct,
  CAST(r.abnormal_rate_pct - r.prev_month_abnormal_rate_pct AS DECIMAL(6,2)) AS abnormal_rate_change_pct
FROM with_lag r
LEFT JOIN {catalog}.silver.calendar c
  ON c.date_key = r.month_date_key
""")

In [0]:
updated_df = spark.table(target_table)

record_count = updated_df.count()
print("Processed Record Count:", record_count)

dbutils.notebook.exit(str(record_count))